Imports and Environment Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Load Data and Define Feature Sets

In [2]:
df_train = pd.read_csv('train.csv')
# SNP sets
priority_snps = [f'SNP{i}' for i in range(1, 76)]
optional_snps = [f'SNP{i}' for i in range(76, 289)]
all_snps = priority_snps + optional_snps

Preprocessing and Feature Selection

In [3]:
X_train = df_train[all_snps].fillna(0)
y_mace = df_train['OUTCOME MACE']
y_severity = df_train['OUTCOME SEVERITY']

# Scale
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

# Chi2 selection
k_features = min(50, X_train_scaled.shape[1])
selector_mace = SelectKBest(chi2, k=k_features)
X_train_mace = selector_mace.fit_transform(X_train_scaled, y_mace)

selector_sev = SelectKBest(chi2, k=k_features)
X_train_sev = selector_sev.fit_transform(X_train_scaled, y_severity)

Internal Model Validation

In [4]:
X_tr_mace, X_val_mace, y_tr_mace, y_val_mace = train_test_split(
    X_train_mace, y_mace, test_size=0.2, random_state=42
)
X_tr_sev, X_val_sev, y_tr_sev, y_val_sev = train_test_split(
    X_train_sev, y_severity, test_size=0.2, random_state=42
)

# Random Forest
rf_mace = RandomForestClassifier(n_estimators=100, random_state=42)
rf_mace.fit(X_tr_mace, y_tr_mace)
print(f"Random Forest Accuracy OUTCOME MACE (train): {accuracy_score(y_val_mace, rf_mace.predict(X_val_mace)):.2%}")

rf_sev = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sev.fit(X_tr_sev, y_tr_sev)
print(f"Random Forest Accuracy OUTCOME SEVERITY (train): {accuracy_score(y_val_sev, rf_sev.predict(X_val_sev)):.2%}")

Random Forest Accuracy OUTCOME MACE (train): 78.57%
Random Forest Accuracy OUTCOME SEVERITY (train): 71.43%


Test Set Preparation

In [5]:
df_test = pd.read_csv('test.csv')
for snp in all_snps:
    if snp not in df_test.columns:
        df_test[snp] = 0

# Reorder columns to match training
X_test_all = df_test[all_snps].fillna(0)
X_test_all = X_test_all[all_snps]

# Scale test set
X_test_scaled = pd.DataFrame(scaler.transform(X_test_all), columns=X_test_all.columns)

# Chi2 selection on test set (use fitted selectors)
X_test_mace = selector_mace.transform(X_test_scaled)
X_test_sev  = selector_sev.transform(X_test_scaled)

Final Predictions and Submission Export

In [6]:
# Predict on test set
y_pred_mace = rf_mace.predict(X_test_mace)
y_pred_sev  = rf_sev.predict(X_test_sev)

# Submission
submission = pd.DataFrame({
    'trustii_id': df_test['trustii_id'],  # make sure this column exists in test.csv
    'OUTCOME MACE': y_pred_mace,
    'OUTCOME SEVERITY': y_pred_sev
})

submission.to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv")

Predictions saved to predictions.csv
